In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
ProjDIR = "/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/" # Change to your project directory
sys.path.insert(1, f'{ProjDIR}/src/')
from CellType_PSY import *
import yaml
with open(ProjDIR + '/config/config.yaml', 'r') as file:
    config = yaml.safe_load(file)

HGNC, ENSID2Entrez, GeneSymbol2Entrez, Entrez2Symbol = LoadGeneINFO()
try:
    os.chdir(f"{ProjDIR}/notebooks_rebuttal/")
    print(f"Current working directory: {os.getcwd()}")
except FileNotFoundError as e:
    print(f"Error: Could not change directory - {e}")
except Exception as e:  
    print(f"Unexpected error: {e}")    

In [ ]:
expression_matrix = config['analysis_types']['Centering']
print(expression_matrix)
HCT_Spec_MAT = pd.read_csv(ProjDIR + expression_matrix, index_col=0)
HCT_Spec_MAT.columns = HCT_Spec_MAT.columns.astype(int)

In [ ]:
GeneCDSLength = pd.read_csv("/home/jw3514/Work/Resources/gencode_v19_longest_cds_per_gene.tsv", delimiter="\t")
BrainSpan = pd.read_csv("/home/jw3514/Work/CellType_Psy/dat2/ExpMatch/BrainSpan.MatchDF.csv", index_col=0)
HumanSC = pd.read_csv("/home/jw3514/Work/CellType_Psy/dat2/ExpMatch/HumanCT.MatchDF.csv", index_col=0)
merged_loeuf_df = pd.read_csv("../dat/gnomad.LOEUF.merged.csv", index_col=0)
merged_loeuf_df.index = merged_loeuf_df.index.astype(int)
PhastCons = pd.read_csv("/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/results/phastCons_scores_all_genes_EntrezOnly.csv", delimiter=",", index_col="EntrezID")

In [ ]:
GeneCDSLength.head(2)

In [ ]:
PhastCons.head(2)

In [ ]:
merged_loeuf_df.head(2)

In [ ]:
BrainSpan.head(2)

In [ ]:
# Inspect the structure of dataframes to understand how to match them
print("HCT_Spec_MAT shape:", HCT_Spec_MAT.shape)
print("HCT_Spec_MAT index (first 5):", HCT_Spec_MAT.index[:5].tolist())
print("\nGeneCDSLength columns:", GeneCDSLength.columns.tolist())
print("GeneCDSLength head:")
print(GeneCDSLength.head())
print("\nBrainSpan columns:", BrainSpan.columns.tolist())
print("BrainSpan index (first 5):", BrainSpan.index[:5].tolist())
print("\nHumanSC columns:", HumanSC.columns.tolist())
print("HumanSC index (first 5):", HumanSC.index[:5].tolist())


In [ ]:
# Create master table
# Step 1: Get all genes from HCT_Spec_MAT and convert to entrez IDs
genes_from_hct = HCT_Spec_MAT.index.tolist()

# Try to convert to entrez IDs - check if index is already entrez IDs, ensemble IDs, or gene symbols
entrez_ids = []
for gene in genes_from_hct:
    entrez_id = None
    # Try as entrez ID first
    try:
        entrez_id = int(gene)
        if entrez_id in Entrez2Symbol:
            entrez_ids.append(entrez_id)
            continue
    except (ValueError, TypeError):
        pass
    
    # Try as ensemble ID
    if gene in ENSID2Entrez:
        entrez_id = ENSID2Entrez[gene]
        if entrez_id:
            entrez_ids.append(entrez_id)
            continue
    
    # Try as gene symbol
    if gene in GeneSymbol2Entrez:
        entrez_id = GeneSymbol2Entrez[gene]
        if entrez_id:
            entrez_ids.append(entrez_id)
            continue
    
    # If no match found, skip or use None
    entrez_ids.append(None)

# Create initial master dataframe with entrez IDs as index
master_table = pd.DataFrame(index=entrez_ids)
master_table = master_table[master_table.index.notna()]  # Remove None values
master_table.index = master_table.index.astype(int)
master_table.index.name = 'EntrezID'

print(f"Total genes in HCT_Spec_MAT: {len(genes_from_hct)}")
print(f"Successfully matched to entrez IDs: {len(master_table)}")
print(f"Master table shape: {master_table.shape}")

In [ ]:
# Step 2: Add CDS length from GeneCDSLength
# GeneCDSLength has gene_id (ensemble ID) and gene_name (gene symbol) columns
print("GeneCDSLength structure:")
print(GeneCDSLength.head())
print("\nGeneCDSLength columns:", GeneCDSLength.columns.tolist())

# Convert to entrez IDs and create CDS dataframe
# Try both gene_id (ensemble ID) and gene_name (gene symbol) for matching
gene_cds_entrez = []
gene_cds_length = []

for idx, row in GeneCDSLength.iterrows():
    entrez_id = None
    cds_length = row['cds_len']
    
    if pd.isna(cds_length):
        continue
    
    # First try: Use gene_id (ensemble ID) - may have version numbers like .10, .5
    gene_id = row['gene_id']
    if pd.notna(gene_id) and isinstance(gene_id, str):
        # Try exact match first
        if gene_id in ENSID2Entrez:
            entrez_id = ENSID2Entrez[gene_id]
        else:
            # Try removing version number (e.g., ENSG00000000003.10 -> ENSG00000000003)
            gene_id_no_version = gene_id.split('.')[0]
            if gene_id_no_version in ENSID2Entrez:
                entrez_id = ENSID2Entrez[gene_id_no_version]
    
    # Second try: Use gene_name (gene symbol) if ensemble ID didn't work
    if entrez_id is None:
        gene_name = row['gene_name']
        if pd.notna(gene_name) and isinstance(gene_name, str):
            if gene_name in GeneSymbol2Entrez:
                entrez_id = GeneSymbol2Entrez[gene_name]
    
    if entrez_id and pd.notna(cds_length):
        gene_cds_entrez.append(entrez_id)
        gene_cds_length.append(cds_length)

# Create CDS length dataframe
if gene_cds_entrez:
    cds_df = pd.DataFrame({'CDS_length': gene_cds_length}, index=gene_cds_entrez)
    cds_df.index = cds_df.index.astype(int)
    cds_df.index.name = 'EntrezID'
    # Remove duplicates if any (keep first occurrence)
    cds_df = cds_df[~cds_df.index.duplicated(keep='first')]
    print(f"\nCDS length data: {len(cds_df)} genes matched")
    print(f"Sample matches: {cds_df.head()}")
else:
    print("\nWarning: Could not match CDS length data. Please check GeneCDSLength structure.")
    cds_df = pd.DataFrame()


In [ ]:
# Step 3: Merge all data into master table
# Merge CDS length
if not cds_df.empty:
    master_table = pd.merge(master_table, cds_df, left_index=True, right_index=True, how='left')
    print(f"CDS length merged: {master_table['CDS_length'].notna().sum()} genes have CDS length")

# Merge BrainSpan WB (assuming index is already entrez ID)
if 'WB' in BrainSpan.columns:
    brainspan_wb = BrainSpan[['WB']].copy()
    brainspan_wb.index = brainspan_wb.index.astype(int)
    brainspan_wb.index.name = 'EntrezID'
    master_table = pd.merge(master_table, brainspan_wb, left_index=True, right_index=True, how='left')
    print(f"BrainSpan WB merged: {master_table['WB'].notna().sum()} genes have BrainSpan WB")
else:
    print("Warning: 'WB' column not found in BrainSpan")

# Merge HumanSC Exp (assuming index is already entrez ID)
if 'Exp' in HumanSC.columns:
    humansc_exp = HumanSC[['Exp']].copy()
    humansc_exp.index = humansc_exp.index.astype(int)
    humansc_exp.index.name = 'EntrezID'
    master_table = pd.merge(master_table, humansc_exp, left_index=True, right_index=True, how='left')
    print(f"HumanSC Exp merged: {master_table['Exp'].notna().sum()} genes have HumanSC Exp")
else:
    print("Warning: 'Exp' column not found in HumanSC")

# Merge LOEUF from merged_loeuf_df (convert float index to int)
if 'LOEUF' in merged_loeuf_df.columns:
    loeuf_data = merged_loeuf_df[['LOEUF']].copy()
    # Convert float index to int
    loeuf_data.index = loeuf_data.index.astype(float).astype(int)
    loeuf_data.index.name = 'EntrezID'
    master_table = pd.merge(master_table, loeuf_data, left_index=True, right_index=True, how='left')
    print(f"LOEUF merged: {master_table['LOEUF'].notna().sum()} genes have LOEUF")
else:
    print("Warning: 'LOEUF' column not found in merged_loeuf_df")

# Merge PhastCons data (mean_phastCons and n_CDS_bases)
phastcons_cols_to_add = []
if 'mean_phastCons' in PhastCons.columns:
    phastcons_cols_to_add.append('mean_phastCons')
if 'n_CDS_bases' in PhastCons.columns:
    phastcons_cols_to_add.append('n_CDS_bases')

if phastcons_cols_to_add:
    phastcons_data = PhastCons[phastcons_cols_to_add].copy()
    phastcons_data.index = phastcons_data.index.astype(int)
    phastcons_data.index.name = 'EntrezID'
    master_table = pd.merge(master_table, phastcons_data, left_index=True, right_index=True, how='left')
    for col in phastcons_cols_to_add:
        print(f"PhastCons {col} merged: {master_table[col].notna().sum()} genes have {col}")
else:
    print("Warning: 'mean_phastCons' or 'n_CDS_bases' columns not found in PhastCons")

print(f"\nFinal master table shape: {master_table.shape}")
print(f"Master table columns: {master_table.columns.tolist()}")
print("\nMaster table head:")


In [ ]:
master_table.head(5)

In [ ]:
# Pairwise correlation scatter plots, now including mean_phastCons
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import pearsonr

# Get the numeric columns for plotting, now including mean_phastCons
plot_vars = ['CDS_length', 'WB', 'Exp', 'LOEUF', 'mean_phastCons']
# Only include mean_phastCons if it exists in the table
plot_vars = [v for v in plot_vars if v in master_table.columns]
plot_data = master_table[plot_vars].copy()

# Remove rows with any NaN values for correlation calculation
plot_data_clean = plot_data.dropna()

print(f"Number of genes with complete data: {len(plot_data_clean)}")
print(f"\nCorrelation matrix (including mean_phastCons):")
corr_matrix = plot_data_clean.corr()
print(corr_matrix)

# Create pairwise scatter plots
variables = plot_vars
n_vars = len(variables)

# Create figure with subplots
fig, axes = plt.subplots(n_vars, n_vars, figsize=(3*n_vars+2, 3*n_vars+2))
fig.suptitle('Pairwise Correlation Scatter Plots (including mean_phastCons)', fontsize=16, y=1.02)

# Plot scatter plots for each pair
for i, var1 in enumerate(variables):
    for j, var2 in enumerate(variables):
        ax = axes[i, j]
        
        if i == j:
            # Diagonal: show histogram
            ax.hist(plot_data_clean[var1].dropna(), bins=50, alpha=0.7, edgecolor='black')
            ax.set_xlabel(var1)
            ax.set_ylabel('Frequency')
            ax.set_title(var1)
        else:
            # Off-diagonal: scatter plot
            x_data = plot_data_clean[var2]
            y_data = plot_data_clean[var1]
            
            # Remove any remaining NaN values
            mask = ~(pd.isna(x_data) | pd.isna(y_data))
            x_clean = x_data[mask]
            y_clean = y_data[mask]
            
            if len(x_clean) > 0:
                ax.scatter(x_clean, y_clean, alpha=0.3, s=10, edgecolors='none')
                
                # Calculate and display correlation
                if len(x_clean) > 1:
                    corr, pval = pearsonr(x_clean, y_clean)
                    ax.text(0.05, 0.95, f'r = {corr:.3f}\np = {pval:.2e}', 
                           transform=ax.transAxes, fontsize=9,
                           verticalalignment='top', bbox=dict(boxstyle='round', 
                           facecolor='white', alpha=0.8))
        
        # Set labels only on outer edges
        if i == n_vars - 1:
            ax.set_xlabel(var2)
        if j == 0:
            ax.set_ylabel(var1)
        
        # Rotate x-axis labels for readability
        ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Also create a correlation heatmap
plt.figure(figsize=(3*n_vars, 2+n_vars))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Mask upper triangle
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1)
plt.title('Correlation Matrix (Lower Triangle, incl. mean_phastCons)', fontsize=14, pad=20)
plt.tight_layout()
plt.show()


In [ ]:
master_table.to_csv("../dat/Variable_2_Match_master_table.csv")

In [ ]:
master_table

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# Scatter plot: mean_phastCons vs LOEUF
plt.figure(figsize=(7, 5))
x = master_table['mean_phastCons']
y = master_table['LOEUF']
mask = (~x.isna()) & (~y.isna())
x = x[mask]
y = y[mask]

sns.scatterplot(x=x, y=y, alpha=0.4, s=18)
plt.xlabel("mean_phastCons")
plt.ylabel("LOEUF")
plt.title("Scatter plot of mean_phastCons vs LOEUF")

# Correlation
corr, pval = pearsonr(x, y)
plt.text(0.05, 0.95, f'r = {corr:.3f}\np = {pval:.2e}', transform=plt.gca().transAxes, 
         fontsize=11, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.show()

In [ ]:
SCZ_GW = pd.read_csv("../dat/GeneWeights/SCZ.top61.nopLI.LGD_Dmis_SameWeight.exclude_Mis2.gw", index_col=0, header=None)
SCZ_Genes = SCZ_GW.index.tolist()

In [ ]:
SCZ_GW = Fil2Dict("../dat/GeneWeights/SCZ.top61.nopLI.LGD_Dmis_SameWeight.exclude_Mis2.gw")
SCZ_Genes = list(SCZ_GW.keys())

In [ ]:
SCZ_Genes

In [ ]:
# Gene matching algorithm using percentiles and kernel weighting
# Convert SCZ_Genes to integers (entrez IDs)
SCZ_Genes = [int(g) for g in SCZ_Genes if str(g).isdigit()]

print(f"Number of SCZ genes: {len(SCZ_Genes)}")
print(f"First 10 SCZ genes: {SCZ_Genes[:10]}")

# Check which SCZ genes are in master_table
SCZ_Genes_in_master = [g for g in SCZ_Genes if g in master_table.index]
print(f"SCZ genes in master table: {len(SCZ_Genes_in_master)}/{len(SCZ_Genes)}")

# Filter master_table to genes with complete data for matching variables
matching_vars = ['CDS_length', 'WB', 'LOEUF', 'mean_phastCons', 'n_CDS_bases']
master_table_complete = master_table[matching_vars].dropna()
print(f"\nGenes with complete matching data: {len(master_table_complete)}")

# Exclude SCZ genes from candidate pool
candidate_genes = master_table_complete.index.difference(SCZ_Genes_in_master).tolist()
print(f"Candidate genes for matching (excluding SCZ genes): {len(candidate_genes)}")


In [ ]:
# Convert variables to percentiles (0-100 scale)
def convert_to_percentiles(df, columns):
    """Convert specified columns to percentiles"""
    df_percentiles = df.copy()
    for col in columns:
        if col in df.columns:
            # Calculate percentiles (0-100 scale)
            df_percentiles[col + '_pct'] = df[col].rank(pct=True) * 100
    return df_percentiles

# Convert matching variables to percentiles
master_table_pct = convert_to_percentiles(master_table_complete, matching_vars)
candidate_table_pct = master_table_pct.loc[candidate_genes]

print("Percentile conversion complete")
print(f"Sample percentile values:")
print(master_table_pct.loc[SCZ_Genes_in_master[:5] if SCZ_Genes_in_master else []])
master_table_pct.to_csv("../dat/Variable_2_Match_master_table_pct.csv")


In [ ]:
master_table_pct

In [ ]:
# Kernel functions for weighting
def uniform_kernel(distance, bandwidth):
    """Uniform kernel: weight = 1 if distance < bandwidth, else 0"""
    return (distance <= bandwidth).astype(float)

def tricubic_kernel(distance, bandwidth):
    """Tricubic kernel: weight = (1 - (d/h)^3)^3 if d < h, else 0"""
    normalized_dist = distance / bandwidth
    mask = normalized_dist < 1.0
    weights = np.zeros_like(normalized_dist)
    weights[mask] = (1 - normalized_dist[mask]**3)**3
    return weights

def calculate_distance_percentile(gene_percentiles, candidate_percentiles):
    """Calculate Euclidean distance in percentile space"""
    # Get percentile columns (gene_percentiles is a Series, candidate_percentiles is a DataFrame)
    pct_cols = [col for col in candidate_percentiles.columns if col.endswith('_pct')]
    
    # Extract values - gene_percentiles is a Series, so use .loc or .values
    gene_pct = gene_percentiles[pct_cols].values  # This will be 1D array
    candidate_pct = candidate_percentiles[pct_cols].values  # This will be 2D array (n_genes x n_vars)
    
    # Calculate Euclidean distance: sqrt(sum((candidate - gene)^2))
    # Broadcasting: candidate_pct (n_genes, n_vars) - gene_pct (n_vars,) -> (n_genes, n_vars)
    distances = np.sqrt(np.sum((candidate_pct - gene_pct)**2, axis=1))
    return distances

def match_gene(gene_id, candidate_table_pct, master_table_pct, 
               kernel='tricubic', bandwidth=10.0, min_candidates=10):
    """
    Match a gene to candidates based on percentile distance and kernel weighting
    
    Parameters:
    -----------
    gene_id : int
        Entrez ID of gene to match
    candidate_table_pct : DataFrame
        Candidate genes with percentile values
    master_table_pct : DataFrame
        All genes with percentile values (for getting target gene values)
    kernel : str
        'uniform' or 'tricubic'
    bandwidth : float
        Bandwidth for kernel (in percentile units)
    min_candidates : int
        Minimum number of candidates required
    
    Returns:
    --------
    matched_gene_id : int or None
        Matched gene ID, or None if no suitable match found
    """
    if gene_id not in master_table_pct.index:
        return None
    
    # Get target gene percentiles
    target_percentiles = master_table_pct.loc[gene_id]
    
    # Calculate distances to all candidates
    distances = calculate_distance_percentile(
        target_percentiles, 
        candidate_table_pct
    )
    
    # Apply kernel
    if kernel == 'uniform':
        weights = uniform_kernel(distances, bandwidth)
    elif kernel == 'tricubic':
        weights = tricubic_kernel(distances, bandwidth)
    else:
        raise ValueError(f"Unknown kernel: {kernel}")
    
    # Get candidate gene IDs
    candidate_ids = candidate_table_pct.index.values
    
    # Filter to candidates with non-zero weight
    valid_mask = weights > 0
    valid_candidates = candidate_ids[valid_mask]
    valid_weights = weights[valid_mask]
    
    if len(valid_candidates) < min_candidates:
        # If not enough candidates, increase bandwidth and try again
        if kernel == 'tricubic':
            # Try with larger bandwidth
            weights = tricubic_kernel(distances, bandwidth * 2)
            valid_mask = weights > 0
            valid_candidates = candidate_ids[valid_mask]
            valid_weights = weights[valid_mask]
        
        if len(valid_candidates) < min_candidates:
            return None
    
    # Normalize weights to probabilities
    if valid_weights.sum() > 0:
        probabilities = valid_weights / valid_weights.sum()
        # Sample one gene based on weights
        matched_idx = np.random.choice(len(valid_candidates), p=probabilities)
        return valid_candidates[matched_idx]
    else:
        return None

# Test matching for one gene
if SCZ_Genes_in_master:
    test_gene = SCZ_Genes_in_master[0]
    print(f"\nTesting matching for gene {test_gene}:")
    print(f"Target gene percentiles:")
    print(master_table_pct.loc[test_gene][[col + '_pct' for col in matching_vars]])
    
    matched = match_gene(test_gene, candidate_table_pct, master_table_pct, 
                        kernel='tricubic', bandwidth=10.0)
    if matched:
        print(f"\nMatched to gene {matched}:")
        print(f"Matched gene percentiles:")
        print(master_table_pct.loc[matched][[col + '_pct' for col in matching_vars]])
        print(f"\nOriginal values comparison:")
        print("Target:", master_table_complete.loc[test_gene][matching_vars].to_dict())
        print("Matched:", master_table_complete.loc[matched][matching_vars].to_dict())


In [ ]:
# Match all SCZ genes
# Set random seed for reproducibility
np.random.seed(42)

# Parameters for matching
kernel_type = 'tricubic'  # or 'uniform'
bandwidth = 10.0  # percentile units

# Match each SCZ gene
SCZ_matched_genes = {}
matching_stats = {'successful': 0, 'failed': 0, 'not_in_master': 0}

for scz_gene in SCZ_Genes_in_master:
    matched = match_gene(scz_gene, candidate_table_pct, master_table_pct,
                        kernel=kernel_type, bandwidth=bandwidth)
    if matched:
        SCZ_matched_genes[scz_gene] = matched
        matching_stats['successful'] += 1
        # Remove matched gene from candidate pool to avoid duplicates
        if matched in candidate_genes:
            candidate_genes.remove(matched)
            candidate_table_pct = master_table_pct.loc[candidate_genes]
    else:
        matching_stats['failed'] += 1

for scz_gene in SCZ_Genes:
    if scz_gene not in SCZ_Genes_in_master:
        matching_stats['not_in_master'] += 1

print(f"Matching complete!")
print(f"Kernel: {kernel_type}, Bandwidth: {bandwidth}")
print(f"\nMatching statistics:")
print(f"  Successful matches: {matching_stats['successful']}")
print(f"  Failed matches: {matching_stats['failed']}")
print(f"  SCZ genes not in master table: {matching_stats['not_in_master']}")
print(f"\nTotal SCZ genes: {len(SCZ_Genes)}")
print(f"Matched genes: {len(SCZ_matched_genes)}")

# Create matched gene set
SCZ_Matched_Genes = list(SCZ_matched_genes.values())
print(f"\nMatched gene set size: {len(SCZ_Matched_Genes)}")
print(f"First 10 matched genes: {SCZ_Matched_Genes[:10]}")


In [ ]:
# Visualize matching quality
# Compare distributions of matching variables between SCZ genes and matched genes

if len(SCZ_matched_genes) > 0:
    # Get data for SCZ and matched genes
    scz_data = master_table_complete.loc[list(SCZ_matched_genes.keys())]
    matched_data = master_table_complete.loc[SCZ_Matched_Genes]
    
    # Create comparison plots
    fig, axes = plt.subplots(1, len(matching_vars), figsize=(5*len(matching_vars), 4))
    if len(matching_vars) == 1:
        axes = [axes]
    
    for i, var in enumerate(matching_vars):
        ax = axes[i]
        
        # Plot distributions
        ax.hist(scz_data[var].dropna(), bins=30, alpha=0.6, label='SCZ Genes', 
               color='red', edgecolor='black')
        ax.hist(matched_data[var].dropna(), bins=30, alpha=0.6, label='Matched Genes', 
               color='blue', edgecolor='black')
        
        ax.set_xlabel(var, fontsize=12)
        ax.set_ylabel('Frequency', fontsize=12)
        ax.set_title(f'Distribution: {var}', fontsize=12)
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('Matching Quality: SCZ vs Matched Genes', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Statistical comparison
    print("\nStatistical comparison (SCZ vs Matched):")
    print("=" * 60)
    for var in matching_vars:
        scz_vals = scz_data[var].dropna()
        matched_vals = matched_data[var].dropna()
        
        from scipy.stats import mannwhitneyu
        stat, pval = mannwhitneyu(scz_vals, matched_vals, alternative='two-sided')
        
        print(f"\n{var}:")
        print(f"  SCZ: mean={scz_vals.mean():.3f}, median={scz_vals.median():.3f}")
        print(f"  Matched: mean={matched_vals.mean():.3f}, median={matched_vals.median():.3f}")
        print(f"  Mann-Whitney U test: p={pval:.4f}")
    
    # Create matching table
    matching_df = pd.DataFrame({
        'SCZ_Gene': list(SCZ_matched_genes.keys()),
        'Matched_Gene': list(SCZ_matched_genes.values())
    })
    
    # Add variable values for comparison
    for var in matching_vars:
        matching_df[f'SCZ_{var}'] = matching_df['SCZ_Gene'].map(master_table_complete[var])
        matching_df[f'Matched_{var}'] = matching_df['Matched_Gene'].map(master_table_complete[var])
        matching_df[f'{var}_diff'] = matching_df[f'SCZ_{var}'] - matching_df[f'Matched_{var}']
    
    print(f"\n\nMatching table (first 10 rows):")
    print(matching_df.head(10))
    
    # Save matching results
    matching_df.to_csv("../dat/SCZ_Matched_Genes.csv", index=False)
    print(f"\nMatching results saved to: ../dat/SCZ_Matched_Genes.csv")
    
    # Save matched gene list
    pd.Series(SCZ_Matched_Genes, name='Matched_Gene').to_csv(
        "../dat/SCZ_Matched_Genes_list.csv", index=False)
    print(f"Matched gene list saved to: ../dat/SCZ_Matched_Genes_list.csv")
else:
    print("No matched genes to visualize")


In [ ]:
candidate_pct = np.array([50, 40])
gene_pct = np.array([60, 30])
distances = np.sqrt(np.sum((candidate_pct - gene_pct)**2))
distances